In [0]:
# Databricks notebook source
# =========================
# Widgets & Config
# =========================
dbutils.widgets.text("snapshot_quarter", "2026Q1")
dbutils.widgets.text("prompt_version", "v4.0.0")
dbutils.widgets.text("endpoint", "databricks-gpt-5-2")
dbutils.widgets.text("test_mode", "true")
dbutils.widgets.text("test_max_groups", "20")
dbutils.widgets.text("rate_limit_seconds", "0.6")
dbutils.widgets.text("max_tokens", "1800")
dbutils.widgets.text("run_scope", "PROD")

dbutils.widgets.text("pval_max", "0.10")
dbutils.widgets.text("abs_coef_min", "0.07")

dbutils.widgets.text("group_dims_to_run", "country")
dbutils.widgets.text("target_y_feature", "07_06_리모컨_사용성")

SNAPSHOT_QUARTER   = dbutils.widgets.get("snapshot_quarter")
PROMPT_VERSION     = dbutils.widgets.get("prompt_version")
ENDPOINT           = dbutils.widgets.get("endpoint")
TEST_MODE          = dbutils.widgets.get("test_mode").lower() == "true"
TEST_MAX_GROUPS    = int(dbutils.widgets.get("test_max_groups"))
RATE_LIMIT_SECONDS = float(dbutils.widgets.get("rate_limit_seconds"))
MAX_TOKENS         = int(dbutils.widgets.get("max_tokens"))
RUN_SCOPE          = dbutils.widgets.get("run_scope").upper()

PVAL_MAX      = float(dbutils.widgets.get("pval_max"))
ABS_COEF_MIN  = float(dbutils.widgets.get("abs_coef_min"))
ABS_COEF_MAX  = float(dbutils.widgets.get("abs_coef_max"))

GDIMS_TO_RUN  = [g.strip() for g in dbutils.widgets.get("group_dims_to_run").split(",") if g.strip()]
TARGET_Y_FEATURE = dbutils.widgets.get("target_y_feature").strip()

THRESHOLDS_META = {
    "p_value_max": PVAL_MAX,
    "abs_coef_min": ABS_COEF_MIN,
    "abs_coef_max": ABS_COEF_MAX
}

print(f"[CONFIG] quarter={SNAPSHOT_QUARTER}, prompt_version={PROMPT_VERSION}, endpoint={ENDPOINT}, run_scope={RUN_SCOPE}")
print(f"[CONFIG] thresholds: pval<{PVAL_MAX}, |coef|∈[{ABS_COEF_MIN},{ABS_COEF_MAX}]")
print(f"[CONFIG] dims: {GDIMS_TO_RUN}, target_y_feature={TARGET_Y_FEATURE}, test_mode={TEST_MODE} (limit={TEST_MAX_GROUPS})")

In [0]:
# =========================
# Imports
# =========================
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql import Row
from datetime import datetime, UTC
import time
import json
import traceback
import re

MODEL_SUMMARY_TABLE = "sandbox.z_jungryo_lee.voc_wls_model_summary"
COEF_SUMMARY_TABLE  = "sandbox.z_jungryo_lee.voc_wls_coef_summary"

TARGET_TABLE = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_v4"
FAILED_TABLE = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_failed_v4"

MAX_RETRIES  = 4
BACKOFF_BASE = 1.8


In [0]:
# =========================
# Prompt
# =========================
SYSTEM_PROMPT = """
You are a senior product strategy planner for TV products.
Return ONLY strict JSON. No preface, no markdown.

Language and style rules:
- Write all outputs in Korean.
- Every sentence must use a formal business tone ending in '~습니다', '~입니다', or equivalent formal style.
- Keep each sentence concise and intuitive for planners and strategists.
- You may use memorable strategic phrasing if it is clear and grounded in the evidence.
- Do not expose raw variable prefixes like '07_02_' or other code-like prefixes in prose.
- If you mention adjusted R-squared, keep the English term exactly as 'adjusted R-squared'.

Analysis rules:
- Base all interpretations only on coefficients, p-values, adjusted R-squared, model p-value, and review volume.
- Always compare group_keys within the given group_dim.
- The goal is not analyst commentary but dashboard-ready planning insight.
- Translate evidence into planning meaning:
  1. what value customers center on
  2. what makes each market distinct
  3. what planning emphasis should differ across markets
- Avoid generic recommendations, filler text, and abstract branding language.

Required output fields:
1. overall_core_message
2. planning_summary
3. country_insight_cards
4. strategy_takeaways

Return ONLY strict JSON following this schema:
{
  "overall_core_message": "전체 뷰의 한 줄 핵심",
  "planning_summary": "기획/전략 관점 요약 멘트",
  "country_insight_cards": [
    {
      "group_key": "Brazil",
      "country_core": "브라질 고객은 리모컨을 연결 경험의 중심으로 인식합니다.",
      "summary_comment": "이 시장에서는 음성 제어와 IoT 연동 계열 가치가 만족도를 설명하는 핵심 축입니다.",
      "key_drivers": [
        {
          "x_feature": "Voice Control",
          "coef": 0.052,
          "direction": "positive",
          "comment": "핵심 동인으로 해석됩니다."
        }
      ],
      "special_points": [
        "다른 시장보다 연결성이 더 중요한 차별 포인트입니다."
      ]
    }
  ],
  "strategy_takeaways": [
    "시장별로 리모컨의 핵심 가치 정의를 다르게 가져갈 필요가 있습니다."
  ]
}
""".strip()

def build_user_prompt(group_dim: str, y_feature: str, thresholds: dict) -> str:
    return f"""
You will receive a JSON payload for one y_feature.

Context:
- group_dim: {group_dim}
- y_feature: {y_feature}

Required output:
- overall_core_message: one-line executive message
- planning_summary: short planning summary
- country_insight_cards:
  - group_key
  - country_core
  - summary_comment
  - key_drivers
  - special_points
- strategy_takeaways

Important writing constraint:
- Every Korean sentence must be in formal business tone such as '~습니다' or '~입니다'.
- Make the writing intuitive for product planners and strategists.
- Do not expose raw prefixes like '07_02_' in output text.
- adjusted R-squared must remain in English if referenced.

Important logic constraint:
- Use only the provided evidence.
- Focus on what each market treats as the core value and what planning emphasis it implies.

Thresholds:
{json.dumps(thresholds, ensure_ascii=False)}

Return ONLY strict JSON following the required schema.
""".strip()

In [0]:
# =========================
# Helpers
# =========================
def rows_to_pylist(lst):
    if not lst:
        return []
    return [dict(x.asDict()) if hasattr(x, "asDict") else dict(x) for x in lst]

def sort_drivers(drivers):
    if not drivers:
        return []
    return sorted(drivers, key=lambda d: abs(d.get("coef", 0.0)), reverse=True)

def json_dumps_safe(obj):
    return json.dumps(obj, ensure_ascii=False)

def safe_float(v, default=None):
    try:
        if v is None:
            return default
        return float(v)
    except Exception:
        return default

def safe_int(v, default=None):
    try:
        if v is None:
            return default
        return int(v)
    except Exception:
        return default

def clean_feature_name(name: str) -> str:
    if not name:
        return ""
    s = str(name)
    s = re.sub(r"^[0-9]+([_./-][0-9]+)*[_./-]*", "", s)
    s = re.sub(r"^[^A-Za-z가-힣]+", "", s)
    s = s.replace("_", " ").strip()
    return s

def cleanse_driver_names(drivers: list) -> list:
    out = []
    for d in drivers or []:
        x = dict(d)
        x["x_feature"] = clean_feature_name(x.get("x_feature"))
        out.append(x)
    return out

def cleanse_comp_stats(comp_stats: list) -> list:
    out = []
    for item in comp_stats or []:
        x = dict(item)
        x["x_feature"] = clean_feature_name(x.get("x_feature"))
        out.append(x)
    return out

def compute_confidence_level(model_stats: list) -> dict:
    if not model_stats:
        return {
            "level": "low",
            "score_basis": {
                "min_adj_r_squared": None,
                "max_model_p_value": None,
                "min_y_obs": None
            }
        }

    adj_r2_list = [safe_float(x.get("adj_r_squared")) for x in model_stats if safe_float(x.get("adj_r_squared")) is not None]
    pval_list   = [safe_float(x.get("model_p_value")) for x in model_stats if safe_float(x.get("model_p_value")) is not None]
    yobs_list   = [safe_int(x.get("y_obs")) for x in model_stats if safe_int(x.get("y_obs")) is not None]

    min_adj_r2 = min(adj_r2_list) if adj_r2_list else None
    max_pval   = max(pval_list) if pval_list else None
    min_y_obs  = min(yobs_list) if yobs_list else None

    if min_adj_r2 is None or max_pval is None or min_y_obs is None:
        level = "low"
    elif min_adj_r2 >= 0.60 and max_pval <= 0.05 and min_y_obs >= 30:
        level = "high"
    elif min_adj_r2 >= 0.30 and max_pval <= 0.10 and min_y_obs >= 10:
        level = "mid"
    else:
        level = "low"

    return {
        "level": level,
        "score_basis": {
            "min_adj_r_squared": min_adj_r2,
            "max_model_p_value": max_pval,
            "min_y_obs": min_y_obs
        }
    }

def confidence_band_from_level(level: str) -> str:
    if level == "high":
        return "Strong"
    if level == "mid":
        return "Moderate"
    return "Caution"


In [0]:
# =========================
# AI Query
# =========================
def call_ai_query(endpoint: str, request_text: str, temperature: float = 0.1, max_tokens: int = 1800):
    df = spark.createDataFrame([(request_text,)], ["p"])
    out = (
        df.select(
            F.expr(f"""
                ai_query(
                    endpoint => '{endpoint}',
                    request  => p,
                    modelParameters => named_struct(
                        'temperature', {float(temperature)},
                        'max_tokens',  {int(max_tokens)}
                    )
                )
            """).alias("json_out")
        ).first()
    )
    return out["json_out"] if out else None

def call_ai_query_with_retry(endpoint: str, system_prompt: str, user_prompt: str, payload_json: str):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            req = f"""
            {system_prompt}

            {user_prompt}

            Input JSON:
            {payload_json}
            """.strip()

            raw = call_ai_query(endpoint, req, temperature=0.1, max_tokens=MAX_TOKENS)
            if not raw:
                raise RuntimeError("Empty response from ai_query")

            clean = raw.replace("```json", "").replace("```", "").strip()
            data = json.loads(clean)
            time.sleep(RATE_LIMIT_SECONDS)
            return data, clean

        except Exception as e:
            wait = BACKOFF_BASE ** (attempt - 1)
            print(f"[WARN] ai_query failed (attempt {attempt}/{MAX_RETRIES}): {repr(e)} -> wait {wait:.1f}s")
            time.sleep(wait)
            last_err = e

    raise RuntimeError(f"ai_query failed after {MAX_RETRIES} retries: {repr(last_err)}")

In [0]:
# =========================
# Load Source Tables
# =========================
model_summary_sdf_org = spark.table(MODEL_SUMMARY_TABLE)
coef_summary_sdf_org  = spark.table(COEF_SUMMARY_TABLE)

if GDIMS_TO_RUN:
    if "all" in [x.lower() for x in GDIMS_TO_RUN]:
        model_summary_sdf = model_summary_sdf_org
        coef_summary_sdf  = coef_summary_sdf_org
    else:
        model_summary_sdf = model_summary_sdf_org.filter(F.col("group_dim").isin(GDIMS_TO_RUN))
        coef_summary_sdf  = coef_summary_sdf_org.filter(F.col("group_dim").isin(GDIMS_TO_RUN))
else:
    model_summary_sdf = model_summary_sdf_org
    coef_summary_sdf  = coef_summary_sdf_org

if TARGET_Y_FEATURE:
    model_summary_sdf = model_summary_sdf.filter(F.col("y_feature") == TARGET_Y_FEATURE)
    coef_summary_sdf  = coef_summary_sdf.filter(F.col("y_feature") == TARGET_Y_FEATURE)

print("[INFO] model_summary rows:", model_summary_sdf.count())
print("[INFO] coef_summary rows :", coef_summary_sdf.count())

# =========================
# Driver Filter
# =========================
drivers_sdf = (
    coef_summary_sdf
    .withColumn("abs_coef", F.abs(F.col("coef")))
    .filter(
        (F.col("p_value") < F.lit(PVAL_MAX)) &
        (F.col("abs_coef") >= F.lit(ABS_COEF_MIN)) 
    )
)

w_abs_desc = W.partitionBy("group_dim", "group_key", "y_feature").orderBy(F.col("abs_coef").desc())

drivers_ranked_sdf = (
    drivers_sdf
    .withColumn("rank_abs", F.dense_rank().over(w_abs_desc))
    .cache()
)

# =========================
# Aggregate
# =========================
ms_agg_sdf = (
    model_summary_sdf
    .groupBy("group_dim", "y_feature")
    .agg(
        F.collect_list(
            F.struct(
                F.col("group_key"),
                F.col("r_squared"),
                F.col("adj_r_squared"),
                F.col("f_statistic"),
                F.col("prob_f").alias("model_p_value"),
                F.col("log_likelihood"),
                F.col("aic"),
                F.col("bic"),
                F.col("y_obs"),
                F.col("cond_no")
            )
        ).alias("model_stats")
    )
)

drivers_agg_sdf = (
    drivers_ranked_sdf
    .select(
        "group_dim", "y_feature", "group_key", "x_feature",
        "coef", "p_value", "t_value", "x_obs", "abs_coef"
    )
    .groupBy("group_dim", "y_feature")
    .agg(
        F.collect_list(
            F.struct(
                F.col("group_key"),
                F.col("x_feature"),
                F.col("coef"),
                F.col("p_value"),
                F.col("t_value"),
                F.col("x_obs"),
                F.col("abs_coef")
            )
        ).alias("drivers_filtered")
    )
)

payload_base_sdf = (
    ms_agg_sdf
    .join(drivers_agg_sdf, ["group_dim", "y_feature"], "left")
    .cache()
)

total_groups = payload_base_sdf.count()
print("[INFO] payload base groups:", total_groups)

if TEST_MODE and total_groups > TEST_MAX_GROUPS:
    payload_base_sdf = payload_base_sdf.limit(TEST_MAX_GROUPS).cache()
    print(f"[INFO] TEST_MODE -> limited to {TEST_MAX_GROUPS}")


In [0]:
# =========================
# Comparative Driver Stats
# =========================
coef_for_compare = (
    coef_summary_sdf
    .select("group_dim", "group_key", "y_feature", "x_feature", "coef")
    .cache()
)

spread_stats_sdf = (
    coef_for_compare
    .groupBy("group_dim", "y_feature", "x_feature")
    .agg(
        F.min("coef").alias("coef_min"),
        F.max("coef").alias("coef_max"),
        F.avg("coef").alias("coef_mean"),
        F.stddev("coef").alias("coef_std")
    )
    .withColumn("coef_spread", F.col("coef_max") - F.col("coef_min"))
)

by_key_list_sdf = (
    coef_for_compare
    .groupBy("group_dim", "y_feature", "x_feature")
    .agg(
        F.collect_list(
            F.struct(F.col("group_key"), F.col("coef"))
        ).alias("by_key_coef_list")
    )
)

comparative_agg_sdf = (
    spread_stats_sdf
    .join(by_key_list_sdf, ["group_dim", "y_feature", "x_feature"], "left")
    .groupBy("group_dim", "y_feature")
    .agg(
        F.collect_list(
            F.struct(
                F.col("x_feature"),
                F.col("coef_min"),
                F.col("coef_max"),
                F.col("coef_mean"),
                F.col("coef_std"),
                F.col("coef_spread"),
                F.col("by_key_coef_list")
            )
        ).alias("x_feature_diff_stats")
    )
)

comp_rows = {(r["group_dim"], r["y_feature"]): r for r in comparative_agg_sdf.collect()}




In [0]:
# =========================
# Payload + Mapping
# =========================
def build_payload(base_row, comp_rows_dict):
    group_dim = base_row["group_dim"]
    y_feature = base_row["y_feature"]

    model_summary = rows_to_pylist(base_row["model_stats"])
    confidence_meta = compute_confidence_level(model_summary)

    payload = {
        "meta": {
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "endpoint": ENDPOINT,
            "run_scope": RUN_SCOPE,
            "generated_at": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "driver_thresholds": THRESHOLDS_META,
            "confidence_rule_basis": confidence_meta["score_basis"]
        },
        "context": {
            "group_dim": group_dim,
            "y_feature_raw": y_feature,
            "y_feature": clean_feature_name(y_feature)
        },
        "model_summary": model_summary,
        "drivers_filtered": cleanse_driver_names(
            sort_drivers(rows_to_pylist(base_row["drivers_filtered"]))
        )[:3],
        "x_feature_diff_stats": []
    }

    return payload

def llm_json_to_result_record(group_dim, y_feature, llm_json, payload_str, computed_confidence):
    return {
        "snapshot_quarter": SNAPSHOT_QUARTER,
        "prompt_version": PROMPT_VERSION,
        "group_dim": group_dim,
        "y_feature": y_feature,
        "y_feature_clean": clean_feature_name(y_feature),
        "overall_core_message": llm_json.get("overall_core_message", ""),
        "planning_summary": llm_json.get("planning_summary", ""),
        "country_insight_cards_json": json_dumps_safe(llm_json.get("country_insight_cards", [])),
        "strategy_takeaways_json": json_dumps_safe(llm_json.get("strategy_takeaways", [])),
        "confidence_level": computed_confidence.get("level", "low"),
        "confidence_band": confidence_band_from_level(computed_confidence.get("level", "low")),
        "raw_llm_json": json_dumps_safe(llm_json),
        "payload_json": payload_str,
        "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
    }


In [0]:
# =========================
# Generate
# =========================
base_rows = payload_base_sdf.collect()
print(f"[INFO] generation targets: {len(base_rows)}")

results = []
failed  = []

for i, r in enumerate(base_rows, 1):
    group_dim = r["group_dim"]
    y_feature = r["y_feature"]

    try:
        payload = build_payload(r, comp_rows)
        payload_str = json.dumps(payload, ensure_ascii=False)

        computed_confidence = compute_confidence_level(payload["model_summary"])

        user_prompt = build_user_prompt(
            group_dim=group_dim,
            y_feature=clean_feature_name(y_feature),
            thresholds=THRESHOLDS_META
        )

        llm_json, raw = call_ai_query_with_retry(
            endpoint=ENDPOINT,
            system_prompt=SYSTEM_PROMPT,
            user_prompt=user_prompt,
            payload_json=payload_str
        )

        results.append(
            llm_json_to_result_record(
                group_dim=group_dim,
                y_feature=y_feature,
                llm_json=llm_json,
                payload_str=payload_str,
                computed_confidence=computed_confidence
            )
        )

        if i % 5 == 0 or i == len(base_rows):
            print(f"[INFO] progress {i}/{len(base_rows)}")

    except Exception as e:
        print(f"[ERROR] ({group_dim}, {y_feature}) failed: {repr(e)}")
        traceback.print_exc()

        failed.append({
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "group_dim": group_dim,
            "y_feature": y_feature,
            "error": repr(e),
            "payload_json": json.dumps(
                {"context": {"group_dim": group_dim, "y_feature": y_feature}},
                ensure_ascii=False
            ),
            "failed_at": datetime.now(UTC)
        })

print(f"[INFO] individual results: success={len(results)}, failed={len(failed)}")

In [0]:
# =========================
# Save Local
# =========================
with open("results_v4.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("[INFO] saved local file: results_v4.json")

In [0]:


# =========================
# Create Tables
# =========================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
snapshot_quarter STRING,
prompt_version STRING,
group_dim STRING,
y_feature STRING,
y_feature_clean STRING,
overall_core_message STRING,
planning_summary STRING,
country_insight_cards_json STRING,
strategy_takeaways_json STRING,
confidence_level STRING,
confidence_band STRING,
raw_llm_json STRING,
payload_json STRING,
generated_at TIMESTAMP
) USING delta
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FAILED_TABLE} (
snapshot_quarter STRING,
prompt_version STRING,
group_dim STRING,
y_feature STRING,
error STRING,
payload_json STRING,
failed_at TIMESTAMP
) USING delta
""")

# =========================
# Merge Success
# =========================
if results:
    out_sdf = spark.createDataFrame([Row(**rec) for rec in results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("overall_core_message").cast("string").alias("overall_core_message"),
        F.col("planning_summary").cast("string").alias("planning_summary"),
        F.col("country_insight_cards_json").cast("string").alias("country_insight_cards_json"),
        F.col("strategy_takeaways_json").cast("string").alias("strategy_takeaways_json"),
        F.col("confidence_level").cast("string").alias("confidence_level"),
        F.col("confidence_band").cast("string").alias("confidence_band"),
        F.col("raw_llm_json").cast("string").alias("raw_llm_json"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )

    out_sdf.createOrReplaceTempView("insight_updates_v4")

    spark.sql(f"""
        MERGE INTO {TARGET_TABLE} AS t
        USING insight_updates_v4 AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
        t.y_feature_clean            = s.y_feature_clean,
        t.overall_core_message       = s.overall_core_message,
        t.planning_summary           = s.planning_summary,
        t.country_insight_cards_json = s.country_insight_cards_json,
        t.strategy_takeaways_json    = s.strategy_takeaways_json,
        t.confidence_level           = s.confidence_level,
        t.confidence_band            = s.confidence_band,
        t.raw_llm_json               = s.raw_llm_json,
        t.payload_json               = s.payload_json,
        t.generated_at               = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
        snapshot_quarter,
        prompt_version,
        group_dim,
        y_feature,
        y_feature_clean,
        overall_core_message,
        planning_summary,
        country_insight_cards_json,
        strategy_takeaways_json,
        confidence_level,
        confidence_band,
        raw_llm_json,
        payload_json,
        generated_at
        )
        VALUES (
        s.snapshot_quarter,
        s.prompt_version,
        s.group_dim,
        s.y_feature,
        s.y_feature_clean,
        s.overall_core_message,
        s.planning_summary,
        s.country_insight_cards_json,
        s.strategy_takeaways_json,
        s.confidence_level,
        s.confidence_band,
        s.raw_llm_json,
        s.payload_json,
        s.generated_at
        )
    """)

    print(f"[INFO] merged success rows: {out_sdf.count()}")

# =========================
# Save Failed
# =========================
if failed:
    failed_sdf = spark.createDataFrame([Row(**rec) for rec in failed]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("error").cast("string").alias("error"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.col("failed_at").cast("timestamp").alias("failed_at")
    )
    failed_sdf.write.mode("append").saveAsTable(FAILED_TABLE)
    print(f"[WARN] failed logged: {failed_sdf.count()}")

# =========================
# Display
# =========================
display(
    spark.table(TARGET_TABLE)
        .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
        .filter(F.col("prompt_version") == PROMPT_VERSION)
        .orderBy(F.desc("generated_at"))
)

In [0]:
%sql
ALTER TABLE sandbox.z_jungryo_lee.voc_wls_dashboard_insight_v4
ADD COLUMNS (
  confidence_level STRING,
  confidence_band STRING
)